# Data Preparation and Feature Engineering

This notebook performs data preprocessing and feature engineering for the campaign donation dataset.

## Tasks:
1. **Data Loading**: Load and merge donor data from CSV files
2. **Preprocessing**: Clean and convert data types (amount, campaign_duration, target_dana)
3. **Feature Engineering**: Calculate progress ratio and create donation amount bins
4. **Probability Distribution**: Calculate probability distribution of donors across bins per campaign
5. **Output**: Create final dataset with 1 row per campaign containing all features


In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import glob
import os
import re
import subprocess
import sys
import json
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Libraries imported successfully!")

# Install gdown if not available
try:
    import gdown
    print("✓ gdown is available")
except ImportError:
    print("Installing gdown...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gdown", "-q"])
    import gdown
    print("✓ gdown installed successfully")


Libraries imported successfully!
✓ gdown is available


## 1. Data Loading

Download JSON files from Google Drive and load them into a single DataFrame.


In [2]:
import os
import json
import pandas as pd
import gdown
from pathlib import Path

# ==========================================
# 1. SETUP & DOWNLOAD
# ==========================================
# Define paths
data_folder = Path('./data')
data_folder.mkdir(exist_ok=True)

output_folder = Path('.')

# Google Drive folder URL for JSON files
google_drive_folder_url = "https://drive.google.com/drive/folders/1BpS_MryIrnks3dXKcbMOBTjpKJ5xEc5j?usp=sharing"
google_drive_folder_id = "1BpS_MryIrnks3dXKcbMOBTjpKJ5xEc5j"

print("Downloading JSON files from Google Drive...")
print(f"Destination: {data_folder.absolute()}")

try:
    gdown.download_folder(
        id=google_drive_folder_id,
        output=str(data_folder),
        quiet=False,
        use_cookies=False
    )
    print(f"\n✓ Files downloaded successfully to {data_folder.absolute()}")
except Exception as e:
    print(f"\n⚠ Error downloading from Google Drive: {e}")
    print("Please ensure files are already in the data folder.")

# Find all JSON files in the data folder
json_files = list(data_folder.glob('donors_*.json'))

if len(json_files) == 0:
    print(f"\n⚠ Warning: No JSON files found in {data_folder.absolute()}")
    raise FileNotFoundError(f"No JSON files found in {data_folder}.")
else:
    print(f"\n✓ Found {len(json_files)} donor JSON files")

print("\nSample files:")
for f in json_files[:5]:
    print(f"  - {f.name}")

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def load_json_file(json_file):
    """
    Load JSON file and extract donor data with metadata.
    """
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Extract metadata
    # Handle structure where metadata might be in 'campaign_metadata' or 'data' or root
    metadata = data.get('campaign_metadata', {})
    if not metadata:
        metadata = data.get('data', {})

    N = metadata.get('N', '')
    campaign_duration = metadata.get('campaign_duration', '')
    target_dana = metadata.get('target_dana', '')

    # Extract donors
    donors = data.get('donors', [])

    # Convert to DataFrame
    if len(donors) > 0:
        df = pd.DataFrame(donors)
        # Add metadata columns to every row
        df['N'] = N
        df['campaign_duration'] = campaign_duration
        df['target_dana'] = target_dana
        return df
    else:
        # Return empty DataFrame with correct columns
        return pd.DataFrame(columns=['name', 'amount', 'time_ago', 'N', 'campaign_duration', 'target_dana'])

# ==========================================
# 3. LOAD & PROCESS (BASED ON FILE)
# ==========================================
all_dataframes = []

print(f"\n{'='*60}")
print("Processing Files (Treating each file as unique source)...")
print(f"{'='*60}")

for json_file in json_files:
    try:
        # Load JSON file
        df = load_json_file(json_file)

        # --- MODIFICATION HERE ---
        # Instead of extracting a slug, we use the FILENAME as the unique ID.
        # This prevents merging different files that share the same campaign name.
        file_identifier = json_file.name

        # Add identifier column
        df['campaign'] = file_identifier

        all_dataframes.append(df)

        print(f"Loaded {json_file.name}: {len(df)} rows")

    except Exception as e:
        print(f"⚠ Error loading {json_file.name}: {e}")
        continue

# ==========================================
# 4. MERGE & SUMMARY
# ==========================================
if len(all_dataframes) > 0:
    df_donors = pd.concat(all_dataframes, ignore_index=True)
    print(f"\n✓ Successfully merged {len(df_donors):,} donor records.")
else:
    raise ValueError("No data loaded from JSON files.")

print(f"\n{'='*60}")
print(f"Data Summary:")
print(f"{'='*60}")
print(f"Total merged rows: {len(df_donors):,}")
# Since we used filename as 'campaign', this counts unique files
print(f"Total unique files (sources): {df_donors['campaign'].nunique()}")
print(f"Columns: {list(df_donors.columns)}")
print(f"{'='*60}")

# Check for required columns
required_cols = ['amount', 'campaign']
missing_cols = [col for col in required_cols if col not in df_donors.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Display sample data
print(f"\nSample data:")
print(df_donors.head())

Destination: /content/data


Retrieving folder contents


Processing file 1sq_HvUw8efoTVJJV_hp273eZ2z6dzURA donors_amaljariyahpembangunanmasjidalmuniri_20251221_125413.json
Processing file 1Yck8M9v4TLUU0Sg_mgH5p1aHuJZXSW_2 donors_bangunmasjidpelosok25_20251221_113953.json
Processing file 15OAhG9crVlsaKJMF1kMpKOSloblyJJSU donors_bangunmesjidmulia127_20251221_091949.json
Processing file 1JxPkQPOdf5tWyCXYBjD5SR-KbZmE34_m donors_bangunmusholadinias_20251221_100648.json
Processing file 1fd3ZGX_47-_dBRyOoalezQLYzbP_bZdW donors_bantupembangunanmasjidfatchulhudadarinol_20251221_124258.json
Processing file 1zKdO5LEXeV1r4D09YpO1d4U_1geP7LZH donors_donasimasjidalfurqon1750_20251221_100244.json
Processing file 1y_In4dSjMOGuWPhWDFM3d7MCOnENnbCl donors_maribangunmasjidberdindingterpalyanghampirroboh_20251221_101146.json
Processing file 1hXjiBlUw-BfThucsbreg24hNzh-H0Iuw donors_maribangunmasjidjatisarigisikdrono_20251221_091657.json
Processing file 1rzlHF1gYfGdktNTOq72dEeWg8yQc1RcL donors_maribantupembangunanmasjiddipedalamansumsel_20251221_125320.json
Proce

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1sq_HvUw8efoTVJJV_hp273eZ2z6dzURA
To: /content/data/donors_amaljariyahpembangunanmasjidalmuniri_20251221_125413.json
100%|██████████| 10.3k/10.3k [00:00<00:00, 18.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Yck8M9v4TLUU0Sg_mgH5p1aHuJZXSW_2
To: /content/data/donors_bangunmasjidpelosok25_20251221_113953.json
100%|██████████| 140k/140k [00:00<00:00, 25.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=15OAhG9crVlsaKJMF1kMpKOSloblyJJSU
To: /content/data/donors_bangunmesjidmulia127_20251221_091949.json
100%|██████████| 13.7k/13.7k [00:00<00:00, 31.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1JxPkQPOdf5tWyCXYBjD5SR-KbZmE34_m
To: /content/data/donors_bangunmusholadinias_20251221_100648.json
100%|██████████| 68.3k/68.3k [00:00<00:00, 46.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1fd3Z


✓ Files downloaded successfully to /content/data

✓ Found 30 donor JSON files

Sample files:
  - donors_bantupembangunanmasjidfatchulhudadarinol_20251221_124258.json
  - donors_masjidbutuhsemen_20251221_101730.json
  - donors_sedekahbangunmasjid712_20251221_100426.json
  - donors_bangunmusholadinias_20251221_100648.json
  - donors_sajadahmesjid_20251221_102158.json

Processing Files (Treating each file as unique source)...
Loaded donors_bantupembangunanmasjidfatchulhudadarinol_20251221_124258.json: 298 rows
Loaded donors_masjidbutuhsemen_20251221_101730.json: 327 rows
Loaded donors_sedekahbangunmasjid712_20251221_100426.json: 182 rows
Loaded donors_bangunmusholadinias_20251221_100648.json: 608 rows
Loaded donors_sajadahmesjid_20251221_102158.json: 166 rows
Loaded donors_setarapulihkanmasjid_20251221_112058.json: 354 rows
Loaded donors_bangunmesjidmulia127_20251221_091949.json: 121 rows
Loaded donors_masjidario_20251221_124111.json: 306 rows
Loaded donors_bangunmasjidpelosok25_20251221


Download completed


## 2. Data Preprocessing

### 2.1 Convert Amount to Numeric


In [3]:
def convert_rupiah_to_number(value):
    """
    Convert Rupiah format string to numeric value.
    Example: 'Rp5.000' -> 5000.0, 'Rp250.000.000' -> 250000000.0
    """
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)

    # Remove 'Rp', dots, commas, and whitespace
    cleaned = str(value).replace('Rp', '').replace('.', '').replace(',', '').strip()

    try:
        return float(cleaned)
    except (ValueError, AttributeError):
        return np.nan

# Convert amount column to numeric
df_donors['amount_numeric'] = df_donors['amount'].apply(convert_rupiah_to_number)

print(f"Amount conversion summary:")
print(f"  - Total rows: {len(df_donors):,}")
print(f"  - Valid amounts: {df_donors['amount_numeric'].notna().sum():,}")
print(f"  - Invalid amounts: {df_donors['amount_numeric'].isna().sum():,}")
print(f"\nSample conversion:")
print(df_donors[['amount', 'amount_numeric']].head(10))

# Check for any invalid conversions
invalid_amounts = df_donors[df_donors['amount_numeric'].isna()]
if len(invalid_amounts) > 0:
    print(f"\nWarning: Found {len(invalid_amounts)} rows with invalid amounts:")
    print(invalid_amounts[['amount', 'campaign']].head())


Amount conversion summary:
  - Total rows: 27,361
  - Valid amounts: 27,361
  - Invalid amounts: 0

Sample conversion:
     amount  amount_numeric
0   Rp4.000         4000.00
1   Rp4.000         4000.00
2  Rp10.000        10000.00
3   Rp5.000         5000.00
4   Rp3.000         3000.00
5  Rp18.000        18000.00
6  Rp16.000        16000.00
7   Rp5.000         5000.00
8  Rp10.000        10000.00
9   Rp7.000         7000.00


### 2.2 Extract Campaign Duration (Integer Days)


In [4]:
def extract_duration_days(duration_str):
    """
    Extract integer days from campaign_duration string.
    Handles formats like:
    - "10 hari lagi" -> 10
    - "100 (days remaining only)" -> 100
    - "0 (days remaining only)" -> 0
    - "10 (days remaining only)" -> 10
    """
    if pd.isna(duration_str):
        return 0

    duration_str = str(duration_str).strip()

    # Try to extract number using regex
    # Match patterns like "10 hari lagi" or "100 (days remaining only)" or "0 (days remaining)"
    match = re.search(r'(\d+)', duration_str)

    if match:
        return int(match.group(1))
    else:
        # If no number found, default to 0
        return 0

# Extract duration for each row (taking first value per campaign since it should be consistent)
# First, get unique campaign durations from metadata
df_donors['campaign_duration_days'] = df_donors['campaign_duration'].apply(extract_duration_days)

print(f"Campaign duration extraction summary:")
print(f"\nUnique duration values found:")
print(df_donors['campaign_duration'].value_counts().head(10))
print(f"\nExtracted days (sample):")
print(df_donors[['campaign_duration', 'campaign_duration_days']].drop_duplicates().head(10))


Campaign duration extraction summary:

Unique duration values found:
campaign_duration
0 (days remaining only)       11566
10 hari lagi                   7020
740 (days remaining only)      1808
1 (days remaining only)        1674
375 (days remaining only)      1086
3417 (days remaining only)      999
71 (days remaining only)        632
64 (days remaining only)        478
13 (days remaining only)        419
14 (days remaining only)        360
Name: count, dtype: int64

Extracted days (sample):
              campaign_duration  campaign_duration_days
0      13 (days remaining only)                      13
298     1 (days remaining only)                       1
1415    2 (days remaining only)                       2
1581   10 (days remaining only)                      10
2056    8 (days remaining only)                       8
2362    0 (days remaining only)                       0
4844    6 (days remaining only)                       6
4914  375 (days remaining only)                     3

### 2.3 Convert Target Dana to Numeric


In [5]:
# Convert target_dana to numeric (using the same function)
df_donors['target_dana_numeric'] = df_donors['target_dana'].apply(convert_rupiah_to_number)

print(f"Target dana conversion summary:")
print(f"  - Valid targets: {df_donors['target_dana_numeric'].notna().sum():,}")
print(f"  - Invalid targets: {df_donors['target_dana_numeric'].isna().sum():,}")
print(f"\nSample conversion:")
print(df_donors[['target_dana', 'target_dana_numeric', 'campaign']].drop_duplicates().head(10))

# Verify target_dana is consistent per campaign
target_check = df_donors.groupby('campaign')['target_dana_numeric'].nunique()
inconsistent_targets = target_check[target_check > 1]
if len(inconsistent_targets) > 0:
    print(f"\nWarning: Found campaigns with inconsistent target_dana:")
    print(inconsistent_targets)
else:
    print(f"\n✓ Target dana is consistent per campaign")


Target dana conversion summary:
  - Valid targets: 27,361
  - Invalid targets: 0

Sample conversion:
          target_dana  target_dana_numeric  \
0     Rp1.000.000.000        1000000000.00   
298     Rp105.000.000         105000000.00   
625      Rp10.000.000          10000000.00   
807     Rp100.000.000         100000000.00   
1415     Rp20.000.000          20000000.00   
1581    Rp100.000.000         100000000.00   
1935     Rp10.000.000          10000000.00   
2056    Rp116.650.000         116650000.00   
2362    Rp100.000.000         100000000.00   
3612    Rp100.000.000         100000000.00   

                                               campaign  
0     donors_bantupembangunanmasjidfatchulhudadarino...  
298        donors_masjidbutuhsemen_20251221_101730.json  
625   donors_sedekahbangunmasjid712_20251221_100426....  
807     donors_bangunmusholadinias_20251221_100648.json  
1415          donors_sajadahmesjid_20251221_102158.json  
1581   donors_setarapulihkanmasjid_20251221_

### 2.4 Clean Data

Remove rows with invalid amounts (keeping only positive, valid amounts).


In [6]:
# Store original count
original_count = len(df_donors)
original_campaigns = df_donors['campaign'].nunique()

# Remove rows with invalid amounts only (keep all campaigns)
df_clean = df_donors.dropna(subset=['amount_numeric'])
df_clean = df_clean[df_clean['amount_numeric'] > 0]

print(f"Data cleaning summary:")
print(f"  - Original rows: {original_count:,}")
print(f"  - Original campaigns: {original_campaigns}")
print(f"  - After cleaning amounts: {len(df_clean):,}")
print(f"  - Removed rows: {original_count - len(df_clean):,}")
print(f"  - Remaining campaigns: {df_clean['campaign'].nunique()}")

# Check campaigns with missing target_dana (will handle at aggregation level)
campaigns_missing_target = df_clean[df_clean['target_dana_numeric'].isna()]['campaign'].unique()
if len(campaigns_missing_target) > 0:
    print(f"\n⚠ Warning: {len(campaigns_missing_target)} campaigns have missing target_dana:")
    print(f"  {list(campaigns_missing_target)[:5]}...")  # Show first 5
    print(f"  These will be handled during aggregation (using 'first' or NaN)")

# Display basic statistics
print(f"\n{'='*60}")
print(f"Dataset Statistics After Cleaning")
print(f"{'='*60}")
print(f"Total donations: {len(df_clean):,}")
print(f"Total amount: Rp {df_clean['amount_numeric'].sum():,.0f}")
print(f"Min donation: Rp {df_clean['amount_numeric'].min():,.0f}")
print(f"Max donation: Rp {df_clean['amount_numeric'].max():,.0f}")
print(f"Average donation: Rp {df_clean['amount_numeric'].mean():,.2f}")
print(f"Median donation: Rp {df_clean['amount_numeric'].median():,.0f}")


Data cleaning summary:
  - Original rows: 27,361
  - Original campaigns: 30
  - After cleaning amounts: 27,361
  - Removed rows: 0
  - Remaining campaigns: 30

Dataset Statistics After Cleaning
Total donations: 27,361
Total amount: Rp 426,372,611
Min donation: Rp 1,000
Max donation: Rp 20,000,000
Average donation: Rp 15,583.22
Median donation: Rp 2,000


## 3. Feature Engineering

### 3.1 Calculate Progress Ratio

Progress = (Total Amount Collected / Target Dana) per campaign


In [7]:
# Calculate total amount collected per campaign
# Use 'first' to get metadata - this handles missing values better
campaign_totals = df_clean.groupby('campaign').agg({
    'amount_numeric': 'sum',
    'target_dana_numeric': 'first',  # Takes first non-null value per campaign
    'campaign_duration_days': 'first'  # Takes first non-null value per campaign
}).reset_index()

campaign_totals.columns = ['campaign', 'total_collected', 'target_dana', 'campaign_duration']

# Fill missing campaign_duration with 0 if needed
campaign_totals['campaign_duration'] = campaign_totals['campaign_duration'].fillna(0).astype(int)

# Calculate progress ratio (handle missing target_dana)
campaign_totals['progress'] = campaign_totals['total_collected'] / campaign_totals['target_dana'].replace(0, np.nan)
campaign_totals['progress'] = campaign_totals['progress'].fillna(0)  # Set progress to 0 if target_dana is missing/0

# Check for campaigns with missing target_dana
missing_target = campaign_totals['target_dana'].isna()
if missing_target.sum() > 0:
    print(f"⚠ Warning: {missing_target.sum()} campaigns have missing target_dana:")
    print(campaign_totals[missing_target][['campaign', 'target_dana', 'total_collected']])

# Also get N (total number of donors per campaign)
campaign_counts = df_clean.groupby('campaign').size().reset_index(name='N')
campaign_totals = campaign_totals.merge(campaign_counts, on='campaign')

print("Progress calculation summary:")
print(f"\nTotal campaigns: {len(campaign_totals)}")
print(f"\nProgress statistics:")
print(f"  - Min progress: {campaign_totals['progress'].min():.4f}")
print(f"  - Max progress: {campaign_totals['progress'].max():.4f}")
print(f"  - Average progress: {campaign_totals['progress'].mean():.4f}")
print(f"  - Median progress: {campaign_totals['progress'].median():.4f}")

# Display sample
print(f"\nSample progress calculations:")
display_cols = ['campaign', 'N', 'total_collected', 'target_dana', 'progress']
print(campaign_totals[display_cols].head(10).to_string(index=False))


Progress calculation summary:

Total campaigns: 30

Progress statistics:
  - Min progress: 0.0000
  - Max progress: 0.6037
  - Average progress: 0.1399
  - Median progress: 0.0850

Sample progress calculations:
                                                                   campaign    N  total_collected   target_dana  progress
           donors_amaljariyahpembangunanmasjidalmuniri_20251221_125413.json   94       1619000.00 1226500000.00      0.00
                          donors_bangunmasjidpelosok25_20251221_113953.json 1250       3989000.00  100000000.00      0.04
                           donors_bangunmesjidmulia127_20251221_091949.json  121       2522000.00   10000000.00      0.25
                            donors_bangunmusholadinias_20251221_100648.json  608      26820000.00  100000000.00      0.27
       donors_bantupembangunanmasjidfatchulhudadarinol_20251221_124258.json  298       8598439.00 1000000000.00      0.01
                       donors_donasimasjidalfurqon1750_20

## 4. Binning and Probability Distribution

### 4.1 Define Hybrid Bins


In [15]:
custom_bins = [0, 10000, 25000, 50000, 100000, float('inf')]

# Label untuk p1, p2, dst
bin_labels = [f'p{i+1}' for i in range(len(custom_bins)-1)]

# Terapkan pd.cut (Value-based binning)
df_clean['amount_bin'] = pd.cut(
    df_clean['amount_numeric'],
    bins=custom_bins,
    labels=bin_labels,
    include_lowest=True, # Agar nilai 0 atau min masuk
    right=True           # Interval (a, b] -> Nilai batas kanan masuk ke bin tersebut
)

# ==========================================
# 1. HITUNG STATISTIK LENGKAP SETIAP BIN
# ==========================================
# Menghitung Mean, Median, Min, Max, dan Count sekaligus
bin_stats = df_clean.groupby('amount_bin', observed=False)['amount_numeric'].agg(['mean', 'median', 'min', 'max', 'count'])

print(f"\n{'='*60}")
print("1. Statistik Lengkap (Min, Max, Median, Mean) per Bin:")
print(f"{'='*60}")

for i, label in enumerate(bin_labels):
    # Ambil range bin untuk display
    lower = custom_bins[i]
    upper = custom_bins[i+1]
    display_lower = lower if i == 0 else lower + 1
    range_str = f"> {lower:,.0f}" if upper == float('inf') else f"{display_lower:,.0f} - {upper:,.0f}"

    # Ambil nilai statistik
    mean_val = bin_stats.loc[label, 'mean']
    median_val = bin_stats.loc[label, 'median']
    min_val = bin_stats.loc[label, 'min']
    max_val = bin_stats.loc[label, 'max']
    count_val = bin_stats.loc[label, 'count']

    # Print hasil
    if pd.notna(mean_val):
        print(f"  {label} ({range_str}):")
        print(f"     -> Jumlah    : {count_val:,} donatur")
        print(f"     -> Min       : Rp {min_val:,.0f}")
        print(f"     -> Max       : Rp {max_val:,.0f}")
        print(f"     -> Median    : Rp {median_val:,.0f}")
        print(f"     -> Rata-rata : Rp {mean_val:,.0f}")
        print("-" * 40)
    else:
        print(f"  {label}: Tidak ada data (NaN)")

# ==========================================
# 2. HITUNG N (MIN, MAX, AVG) DARI 30 FILE
# ==========================================
print(f"\n{'='*60}")
print("2. Statistik N (Jumlah Donatur per Campaign):")
print(f"{'='*60}")

# Hitung jumlah donatur per campaign (berdasarkan nama file/campaign)
donors_per_campaign = df_clean['campaign'].value_counts()

n_min = donors_per_campaign.min()
n_max = donors_per_campaign.max()
n_avg = donors_per_campaign.mean()
total_campaigns = len(donors_per_campaign)

print(f"  Total Campaign Terbaca : {total_campaigns}")
print(f"  ------------------------")
print(f"  N Min (Terkecil)       : {n_min}")
print(f"  N Max (Terbesar)       : {n_max}")
print(f"  N Avg (Rata-rata)      : {n_avg:,.2f}")

# (Opsional) Tampilkan detail campaign dengan N min dan max
min_camp = donors_per_campaign.idxmin()
max_camp = donors_per_campaign.idxmax()
print(f"  ------------------------")
print(f"  Campaign Tersepi       : {min_camp} ({n_min} donatur)")
print(f"  Campaign Teramai       : {max_camp} ({n_max} donatur)")

# ==========================================
# PRINT DISTRIBUSI (FORMAT LAMA UNTUK CEK)
# ==========================================
print(f"\n{'='*60}")
print("Distribution per Bin (Persentase):")
print(f"{'='*60}")
bin_counts = df_clean['amount_bin'].value_counts().sort_index()
bin_proportions = bin_counts / len(df_clean)

for bin_label in bin_labels:
    count = bin_counts.get(bin_label, 0)
    prop = bin_proportions.get(bin_label, 0)
    print(f"  {bin_label}: {count:6,} donors ({prop*100:5.2f}%)")


1. Statistik Lengkap (Min, Max, Median, Mean) per Bin:
  p1 (0 - 10,000):
     -> Jumlah    : 22,860 donatur
     -> Min       : Rp 1,000
     -> Max       : Rp 10,000
     -> Median    : Rp 2,000
     -> Rata-rata : Rp 3,136
----------------------------------------
  p2 (10,001 - 25,000):
     -> Jumlah    : 2,068 donatur
     -> Min       : Rp 10,020
     -> Max       : Rp 25,000
     -> Median    : Rp 20,000
     -> Rata-rata : Rp 19,406
----------------------------------------
  p3 (25,001 - 50,000):
     -> Jumlah    : 1,561 donatur
     -> Min       : Rp 25,172
     -> Max       : Rp 50,000
     -> Median    : Rp 50,000
     -> Rata-rata : Rp 45,707
----------------------------------------
  p4 (50,001 - 100,000):
     -> Jumlah    : 534 donatur
     -> Min       : Rp 50,429
     -> Max       : Rp 100,000
     -> Median    : Rp 100,000
     -> Rata-rata : Rp 95,382
----------------------------------------
  p5 (> 100,000):
     -> Jumlah    : 338 donatur
     -> Min       : Rp 1

### 4.2 Calculate Probability Distribution per Campaign

For each campaign, count donors in each bin and convert to proportions (probabilities).


In [9]:
# Group by campaign and bin, then count
campaign_bin_counts = df_clean.groupby(['campaign', 'amount_bin']).size().reset_index(name='count')

# Pivot to get one row per campaign with columns for each bin
campaign_bin_pivot = campaign_bin_counts.pivot(index='campaign', columns='amount_bin', values='count').fillna(0)

# Ensure all bin columns exist (some campaigns might not have donors in certain bins)
for label in bin_labels:
    if label not in campaign_bin_pivot.columns:
        campaign_bin_pivot[label] = 0

# Reorder columns to match bin_labels order
campaign_bin_pivot = campaign_bin_pivot[bin_labels]

# Calculate total donors per campaign (N) for verification
campaign_bin_pivot['N_calc'] = campaign_bin_pivot[bin_labels].sum(axis=1)

print("Campaign bin counts (first 5 campaigns):")
print(campaign_bin_pivot.head())

# Calculate probabilities (proportions) for each campaign
campaign_probs = campaign_bin_pivot.copy()
campaign_probs['N_calc'] = campaign_bin_pivot['N_calc']  # Store N before converting to probabilities

# Convert counts to probabilities (proportions)
# Handle division by zero: if N_calc is 0, probabilities remain 0
for label in bin_labels:
    campaign_probs[label] = campaign_probs[label].div(campaign_probs['N_calc'].replace(0, np.nan)).fillna(0)

# Drop the N_calc column (we'll use the actual N from campaign_totals)
campaign_probs = campaign_probs.drop(columns=['N_calc'])

# Rename columns to p1, p2, etc. (they should already have these names)
print(f"\nProbability distribution (first 5 campaigns):")
print(campaign_probs.head())

# Verify probabilities sum to 1 for each campaign
prob_sums = campaign_probs[bin_labels].sum(axis=1)
print(f"\nProbability sum verification:")
print(f"  - Min sum: {prob_sums.min():.6f}")
print(f"  - Max sum: {prob_sums.max():.6f}")
print(f"  - Average sum: {prob_sums.mean():.6f}")
print(f"  - All sums equal to 1? {np.allclose(prob_sums, 1.0)}")


Campaign bin counts (first 5 campaigns):
amount_bin                                            p1  p2  p3  p4  p5  \
campaign                                                                   
donors_amaljariyahpembangunanmasjidalmuniri_202...    73  12   5   3   1   
donors_bangunmasjidpelosok25_20251221_113953.json   1227  15   8   0   0   
donors_bangunmesjidmulia127_20251221_091949.json      85  11  16   5   4   
donors_bangunmusholadinias_20251221_100648.json      491  71  31   9   6   
donors_bantupembangunanmasjidfatchulhudadarinol...   204  37  32  14  11   

amount_bin                                          N_calc  
campaign                                                    
donors_amaljariyahpembangunanmasjidalmuniri_202...      94  
donors_bangunmasjidpelosok25_20251221_113953.json     1250  
donors_bangunmesjidmulia127_20251221_091949.json       121  
donors_bangunmusholadinias_20251221_100648.json        608  
donors_bantupembangunanmasjidfatchulhudadarinol...     298  

/tmp/ipython-input-1759743757.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  campaign_bin_counts = df_clean.groupby(['campaign', 'amount_bin']).size().reset_index(name='count')


## 5. Final Output Formatting

Merge all features into a single DataFrame with 1 row per campaign.


In [10]:
# Merge campaign totals (N, campaign_duration, target_dana, progress) with probabilities
df_final = campaign_totals[['campaign', 'N', 'campaign_duration', 'target_dana', 'progress']].copy()

# Merge with probabilities using outer join to ensure all campaigns are included
df_final = df_final.merge(campaign_probs.reset_index(), on='campaign', how='outer')

# Fill missing probability values with 0 (for campaigns that might not have donations in certain bins)
prob_cols = [col for col in df_final.columns if col.startswith('p')]
df_final[prob_cols] = df_final[prob_cols].fillna(0)

# Ensure we have all campaigns
print(f"\nCampaign count verification:")
print(f"  - Campaigns in totals: {len(campaign_totals)}")
print(f"  - Campaigns in probabilities: {len(campaign_probs)}")
print(f"  - Campaigns in final: {len(df_final)}")

# Fill missing N with 0 (shouldn't happen, but just in case)
df_final['N'] = df_final['N'].fillna(0).astype(int)

# Ensure campaign_duration is integer
df_final['campaign_duration'] = df_final['campaign_duration'].fillna(0).astype(int)

# Ensure target_dana is numeric (float) - fill NaN with 0 for now
df_final['target_dana'] = df_final['target_dana'].fillna(0).astype(float)

# Ensure progress is numeric
df_final['progress'] = df_final['progress'].fillna(0).astype(float)

# Reorder columns: campaign, N, campaign_duration, target_dana, progress, p1, p2, ..., pn
column_order = ['campaign', 'N', 'campaign_duration', 'target_dana', 'progress'] + bin_labels
df_final = df_final[column_order]

print("Final dataset structure:")
print(f"  - Shape: {df_final.shape}")
print(f"  - Columns: {list(df_final.columns)}")
print(f"\nFinal dataset preview:")
print(df_final.head(10).to_string(index=False))

print(f"\n{'='*60}")
print("Dataset Summary")
print(f"{'='*60}")
print(f"Total campaigns: {len(df_final)}")
print(f"Total donors (N): {df_final['N'].sum():,}")
print(f"\nCampaign duration statistics:")
print(f"  - Min: {df_final['campaign_duration'].min()} days")
print(f"  - Max: {df_final['campaign_duration'].max()} days")
print(f"  - Mean: {df_final['campaign_duration'].mean():.2f} days")
print(f"\nTarget dana statistics:")
print(f"  - Min: Rp {df_final['target_dana'].min():,.0f}")
print(f"  - Max: Rp {df_final['target_dana'].max():,.0f}")
print(f"  - Mean: Rp {df_final['target_dana'].mean():,.0f}")
print(f"\nProgress statistics:")
print(f"  - Min: {df_final['progress'].min():.6f}")
print(f"  - Max: {df_final['progress'].max():.6f}")
print(f"  - Mean: {df_final['progress'].mean():.6f}")



Campaign count verification:
  - Campaigns in totals: 30
  - Campaigns in probabilities: 30
  - Campaigns in final: 30
Final dataset structure:
  - Shape: (30, 10)
  - Columns: ['campaign', 'N', 'campaign_duration', 'target_dana', 'progress', 'p1', 'p2', 'p3', 'p4', 'p5']

Final dataset preview:
                                                                   campaign    N  campaign_duration   target_dana  progress   p1   p2   p3   p4   p5
           donors_amaljariyahpembangunanmasjidalmuniri_20251221_125413.json   94                 41 1226500000.00      0.00 0.78 0.13 0.05 0.03 0.01
                          donors_bangunmasjidpelosok25_20251221_113953.json 1250                  0  100000000.00      0.04 0.98 0.01 0.01 0.00 0.00
                           donors_bangunmesjidmulia127_20251221_091949.json  121                 13   10000000.00      0.25 0.70 0.09 0.13 0.04 0.03
                            donors_bangunmusholadinias_20251221_100648.json  608                  1  10000

### 5.1 Verify Data Quality


In [11]:
# Check for any missing values
print("Missing values check:")
missing = df_final.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("  ✓ No missing values")

# Verify probability columns sum to approximately 1 for each campaign
prob_cols = [col for col in df_final.columns if col.startswith('p')]
prob_sums = df_final[prob_cols].sum(axis=1)

print(f"\nProbability sum verification:")
print(f"  - All probabilities sum to ~1? {np.allclose(prob_sums, 1.0, atol=1e-6)}")
if not np.allclose(prob_sums, 1.0, atol=1e-6):
    print(f"  - Campaigns with sum != 1:")
    invalid = df_final[~np.isclose(prob_sums, 1.0, atol=1e-6)]['campaign'].tolist()
    print(f"    {invalid}")

# Check that all probability values are between 0 and 1
print(f"\nProbability value range check:")
for col in prob_cols:
    min_val = df_final[col].min()
    max_val = df_final[col].max()
    if min_val < 0 or max_val > 1:
        print(f"  ✗ {col}: min={min_val}, max={max_val}")
    else:
        print(f"  ✓ {col}: min={min_val:.6f}, max={max_val:.6f}")

print(f"\n✓ Data quality checks completed")


Missing values check:
  ✓ No missing values

Probability sum verification:
  - All probabilities sum to ~1? False
  - Campaigns with sum != 1:
    ['donors_amaljariyahpembangunanmasjidalmuniri_20251221_125413.json', 'donors_bangunmasjidpelosok25_20251221_113953.json', 'donors_bangunmesjidmulia127_20251221_091949.json', 'donors_bangunmusholadinias_20251221_100648.json', 'donors_bantupembangunanmasjidfatchulhudadarinol_20251221_124258.json', 'donors_donasimasjidalfurqon1750_20251221_100244.json', 'donors_maribangunmasjidberdindingterpalyanghampirroboh_20251221_101146.json', 'donors_maribangunmasjidjatisarigisikdrono_20251221_091657.json', 'donors_maribantupembangunanmasjiddipedalamansumsel_20251221_125320.json', 'donors_masjidalakbarsorong_20251221_101252.json', 'donors_masjidario_20251221_124111.json', 'donors_masjidbutuhsemen_20251221_101730.json', 'donors_masjidkemasjid_20251221_025520.json', 'donors_masjidpelosoktimur_20251221_102949.json', 'donors_masjidrumahtahfiz_20251221_124621.j

## 6. Save Results

Save the final processed dataset to CSV file.


In [13]:
# Save to CSV
output_file = output_folder / 'data_preparation_result.csv'
df_final.to_csv(output_file, index=False)

print(f"✓ Dataset saved successfully!")
print(f"  - File: {output_file}")
print(f"  - Shape: {df_final.shape}")
print(f"  - Size: {os.path.getsize(output_file) / 1024:.2f} KB")

# Display final summary
print(f"\n{'='*60}")
print("FINAL DATASET SUMMARY")
print(f"{'='*60}")
print(f"Total campaigns: {len(df_final)}")
print(f"Expected campaigns: ~30")
if len(df_final) < 30:
    print(f"⚠ Warning: Only {len(df_final)} campaigns found. Expected ~30.")
    print(f"  Please check if all CSV files were loaded correctly.")
elif len(df_final) > 30:
    print(f"✓ Found {len(df_final)} campaigns (more than expected)")
else:
    print(f"✓ Found exactly {len(df_final)} campaigns as expected")
print(f"\nColumns in final dataset:")
for i, col in enumerate(df_final.columns, 1):
    print(f"  {i:2d}. {col}")

# Show a few example rows
print(f"\nExample rows (first 3 campaigns):")
print(df_final.head(3).to_string(index=False))


✓ Dataset saved successfully!
  - File: data_preparation_result.csv
  - Shape: (30, 10)
  - Size: 5.28 KB

FINAL DATASET SUMMARY
Total campaigns: 30
Expected campaigns: ~30
✓ Found exactly 30 campaigns as expected

Columns in final dataset:
   1. campaign
   2. N
   3. campaign_duration
   4. target_dana
   5. progress
   6. p1
   7. p2
   8. p3
   9. p4
  10. p5

Example rows (first 3 campaigns):
                                                        campaign    N  campaign_duration   target_dana  progress   p1   p2   p3   p4   p5
donors_amaljariyahpembangunanmasjidalmuniri_20251221_125413.json   94                 41 1226500000.00      0.00 0.78 0.13 0.05 0.03 0.01
               donors_bangunmasjidpelosok25_20251221_113953.json 1250                  0  100000000.00      0.04 0.98 0.01 0.01 0.00 0.00
                donors_bangunmesjidmulia127_20251221_091949.json  121                 13   10000000.00      0.25 0.70 0.09 0.13 0.04 0.03
